In [ ]:
# Example 1: Top-level package surface — confirm the primary client classes are exposed.
import fireworks

# Common public names that fireworks_ai exports at the top level.
public_names = [n for n in dir(fireworks) if not n.startswith('_')]
print(f"fireworks public symbols (first 30): {public_names[:30]}")
print(f"total public symbols: {len(public_names)}")

assert len(public_names) > 0, "fireworks should expose at least one public symbol"


In [ ]:
# Example 2: Client construction (no network call).
# The 1.x SDK exports the client from the TOP-LEVEL `fireworks` package. `fireworks.client` is a
# deprecated back-compat shim for the 0.19.x import path; it re-exports the very same objects.
import fireworks
from fireworks import Fireworks

client = Fireworks(api_key="dummy-test-key-not-real")
print(f"client type: {type(client).__name__}")
print(f"client module: {type(client).__module__}")

# The client should expose chat / completions namespaces (mirrors openai layout).
for attr in ("chat", "completions", "embeddings"):
    has = hasattr(client, attr)
    print(f"  client.{attr}: {'present' if has else 'absent'}")

assert isinstance(client, Fireworks)

# The deprecated 0.19.x import path must still resolve to the SAME class objects.
import fireworks.client as fw_client_shim

assert fw_client_shim.Fireworks is fireworks.Fireworks
assert fw_client_shim.AsyncFireworks is fireworks.AsyncFireworks
print(f"fireworks.client shim re-exports: {sorted(fw_client_shim.__all__)}")

In [ ]:
# Example 3: AsyncFireworks construction — async client should also instantiate locally.
from fireworks import AsyncFireworks

aclient = AsyncFireworks(api_key="dummy-test-key-not-real")
print(f"async client type: {type(aclient).__name__}")
print(f"async client module: {type(aclient).__module__}")

# Async client should mirror sync client's attribute surface.
for attr in ("chat", "completions", "embeddings"):
    has = hasattr(aclient, attr)
    print(f"  aclient.{attr}: {'present' if has else 'absent'}")

assert isinstance(aclient, AsyncFireworks)


In [ ]:
# Example 4: Error class hierarchy. In fireworks-ai 1.x the error types are exported from the
# TOP-LEVEL `fireworks` package (they are defined in `fireworks._exceptions`); the 0.19.x
# `fireworks.client.error` module is GONE, and the deprecated `fireworks.client` shim re-exports
# only the clients. Confirm every error type is present, Exception-derived, and correctly nested.
import fireworks

status_errors = ["BadRequestError", "AuthenticationError", "PermissionDeniedError", "NotFoundError",
                 "ConflictError", "UnprocessableEntityError", "RateLimitError", "InternalServerError"]
base_errors = ["FireworksError", "APIError", "APIStatusError", "APIConnectionError",
               "APITimeoutError", "APIResponseValidationError"]
for name in base_errors + status_errors:
    cls = getattr(fireworks, name)
    assert isinstance(cls, type) and issubclass(cls, Exception), f"{name} is not an Exception subclass"
    # `->`, not `:`, so the printed line does not read like an uncaught exception.
    print(f"  {name} -> subclass of {cls.__mro__[1].__name__}")

# The hierarchy edges callers actually branch on.
assert issubclass(fireworks.APIError, fireworks.FireworksError)
assert issubclass(fireworks.APIStatusError, fireworks.APIError)
assert issubclass(fireworks.APIResponseValidationError, fireworks.APIError)
assert issubclass(fireworks.APIConnectionError, fireworks.APIError)
assert issubclass(fireworks.APITimeoutError, fireworks.APIConnectionError)
for name in status_errors:
    assert issubclass(getattr(fireworks, name), fireworks.APIStatusError), name

print(f"\nall {len(base_errors) + len(status_errors)} error classes present and correctly derived")

In [ ]:
# Example 5: an invalid API key is refused with a FireworksError (live call)
from fireworks import Fireworks, FireworksError

client = Fireworks(api_key="obviously-invalid-key-00000000")

raised = None
try:
    # Tiny request - we expect this to fail, NOT to succeed.
    client.chat.completions.create(
        model="accounts/fireworks/models/llama-v3p1-8b-instruct",
        messages=[{"role": "user", "content": "hi"}],
        max_tokens=1,
    )
except BaseException as e:
    raised = e
    print(f"raised: {type(e).__name__}: {str(e)[:200]}")

assert raised is not None, "calling Fireworks API with an invalid key must raise"
assert isinstance(raised, FireworksError), f"expected a FireworksError, got {type(raised).__name__}"